In [1]:
import os
import kagglehub
import shutil

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
print("Fichiers téléchargés dans :", path)

# Dossier de destination
dest = "data"
#os.makedirs(dest, exist_ok=True)

# Copier tous les fichiers du dossier téléchargé vers /data
#for fichier in os.listdir(path):
#    src_file = os.path.join(path, fichier)
#    dst_file = os.path.join(dest, fichier)
#    shutil.copy2(src_file, dst_file)

print("Fichiers copiés dans :", dest)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fichiers téléchargés dans : /home/onyxia/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques
Fichiers copiés dans : data


In [1]:
import pandas as pd
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import importlib
import preprocessing as p
importlib.reload(p)

df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

In [2]:
X=df_train.drop(columns=['SalePrice'])
y=df_train['SalePrice']
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.2,random_state=42)

In [3]:
test_ids = df_test['Id'].copy()
X_train,power_transform,imputer=p.preprocessing(X_train)
X_valid,_,_=p.preprocessing(X_valid,power_transform,imputer)
df_test,_,_=p.preprocessing(df_test,power_transform,imputer)

In [4]:
model = LinearRegression()
model.fit(X_train, np.log1p(y_train))

y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(np.log1p(y_valid), y_pred))
print(rmse)

2.1884903357584227


In [5]:
erreurs = np.abs(y_pred - np.log1p(y_valid).values)
idx = np.argsort(erreurs)[-5:]   # les 5 pires
print("pred :", y_pred[idx])
print("vrai :", np.log1p(y_valid).values[idx])
print("écart:", erreurs[idx])

print(X_valid.describe().loc[['min','max']].T.sort_values('max').tail())
print(X_valid.describe().loc[['min','max']].T.sort_values('min').head())

print(X_train['Utilities'].max(), X_train['Functional'].max())

pred : [ 9.42570171 10.1995874  10.4180644   9.84271959 10.81938781]
vrai : [11.8706069  12.64915783 12.88664358 12.32386013 13.32392858]
écart: [2.44490519 2.44957043 2.46857918 2.48114053 2.50454077]
                      min     max
YearBuilt     1880.000000  2009.0
GarageYrBlt   1900.961278  2009.0
YearRemodAdd  1950.000000  2010.0
YrSold        2006.000000  2010.0
BsmtUnfSF        0.000000  2042.0
              min       max
BsmtQual      0.0  5.000000
BsmtExposure  0.0  4.000000
BsmtFinType1  0.0  6.000000
MasVnrArea    0.0  3.706729
FullBath      0.0  3.000000
4.0 8.0


In [6]:
model.fit(X_train, np.log1p(y_train))
print("intercept :", model.intercept_)
print("moyenne y :", np.log1p(y_train).mean())

y_pred_train = model.predict(X_train)
print("pred train :", y_pred_train[:3])
print("vrai  train:", np.log1p(y_train).values[:3])

intercept : 5.092127879076541
moyenne y : 12.030658310971573
pred train : [11.84576372 12.07941781 11.43157364]
vrai  train: [11.88449592 12.08954445 11.3504183 ]


In [7]:
for col in ['GrLivArea', 'LotArea', '1stFlrSF']:
    print(col, "| train:", round(X_train[col].mean(),2), round(X_train[col].std(),2),
                "| valid:", round(X_valid[col].mean(),2), round(X_valid[col].std(),2))

GrLivArea | train: 7.09 0.31 | valid: 7.05 0.33
LotArea | train: 9.24 0.53 | valid: 9.18 0.53
1stFlrSF | train: 6.41 0.26 | valid: 6.38 0.26


In [8]:
print("Utilities max:", X_train['Utilities'].max(), X_valid['Utilities'].max())
print("colonnes:", X_train.shape[1], X_valid.shape[1])
print("même ordre:", (X_train.columns == X_valid.columns).all())


Utilities max: 4.0 4.0
colonnes: 204 204
même ordre: True
